# 🧩 Notebook 02 — Clustering benchmark

**Goal:** segment credit-card customers using three clustering algorithms and compare them.

We will:
1. Load and preprocess the **Credit Card Customers (CC General)** dataset (8,950 × 17).
2. Pick the right **k** for K-Means using elbow + silhouette diagnostics.
3. Run **K-Means**, **Agglomerative Hierarchical**, and **DBSCAN**.
4. Compare them on Silhouette, Davies-Bouldin, and Calinski-Harabasz scores.
5. Visualize clusters in 2D via PCA.
6. Profile each cluster (mean feature values) for the report's narrative.

All figures save to `results/figures/` with prefix `02_clustering_`.
All tables save to `results/tables/` with the same prefix.


## ⚙️ Step 1 — Setup

In [ ]:
# Clone the project repository
!git clone https://github.com/faresalotaibi888-gif/Data_Mining_HW2.git 2>/dev/null || (cd Data_Mining_HW2 && git pull)
%cd Data_Mining_HW2

# Install dependencies
!pip install -q pandas matplotlib seaborn scikit-learn scipy


## 📥 Step 2 — Load the CC General dataset

The CC General dataset is hosted on Kaggle and requires manual download.

**Option A** — if you've already committed `CC_GENERAL.csv` to the repo's `data/` folder, this cell will load it.

**Option B** — if not, you'll be prompted to upload it from your computer.

Download the file from: https://www.kaggle.com/datasets/arjunbhasin2013/ccdata


In [ ]:
import os
from pathlib import Path

DATA_FILE = Path('data/CC_GENERAL.csv')

if DATA_FILE.exists():
    print(f"✅ Found existing {DATA_FILE} — loading.")
else:
    print(f"⚠️  {DATA_FILE} not found.")
    print("Please upload CC_GENERAL.csv from your computer.")
    from google.colab import files
    uploaded = files.upload()
    # Move the uploaded file into data/
    for filename in uploaded:
        os.makedirs('data', exist_ok=True)
        os.rename(filename, DATA_FILE)
    print(f"✅ Saved to {DATA_FILE}")

# Now load it through our standard data_loader
import sys
sys.path.append('src')
from data_loader import load_cc_general, dataset_summary

df = load_cc_general()
dataset_summary(df, "CC General — Credit Card Customers")


## 🧹 Step 3 — Preprocess

Three steps inside `preprocess_cc_general`:
1. Drop the `CUST_ID` column (identifier — no predictive value).
2. Median-impute any missing values.
3. Standardize all features (essential for distance-based clustering).


In [ ]:
from preprocessing import preprocess_cc_general

X_scaled, feature_names, scaler = preprocess_cc_general(df)
print(f"Preprocessed shape: {X_scaled.shape}")
print(f"Feature count    : {len(feature_names)}")
print(f"First 5 features : {feature_names[:5]}")


## 🎯 Step 4 — Choose k for K-Means

Run K-Means for k = 2…10 and plot two diagnostic curves:
* **Elbow plot** (inertia) — pick k at the bend.
* **Silhouette score** — pick k at the peak.


In [ ]:
from clustering import kmeans_diagnostics
from evaluation import plot_elbow, plot_silhouette_scores

diag = kmeans_diagnostics(X_scaled, k_min=2, k_max=10)

# Side-by-side diagnostic plots, both saved with our naming convention
plot_elbow(
    diag.k_values, diag.inertias,
    filename='02_clustering_elbow_method',
)
plot_silhouette_scores(
    diag.k_values, diag.silhouettes,
    filename='02_clustering_silhouette_kmeans',
)


**Pick k**: usually the elbow lies at 3 or 4 and the silhouette peaks similarly. The cell below picks the k with the **highest silhouette score** automatically — feel free to override.

In [ ]:
# Pick k with the highest silhouette score (objective rule)
best_k = diag.k_values[diag.silhouettes.index(max(diag.silhouettes))]
print(f"Chosen k = {best_k}  (highest silhouette = {max(diag.silhouettes):.3f})")


## 🔵 Step 5 — Run K-Means with chosen k

In [ ]:
from clustering import run_kmeans, compute_cluster_metrics

kmeans_labels, kmeans_model = run_kmeans(X_scaled, n_clusters=best_k)
kmeans_metrics = compute_cluster_metrics(X_scaled, kmeans_labels, 'K-Means')
print(kmeans_metrics)


## 🌲 Step 6 — Agglomerative Hierarchical Clustering

First show the dendrogram (helps the report explain *how* hierarchical clustering builds its tree), then fit with the same `k` chosen above.


In [ ]:
from evaluation import plot_dendrogram

# Sample down to ~500 points for a readable dendrogram (full 8,950 is unreadable)
import numpy as np
rng = np.random.default_rng(42)
idx = rng.choice(len(X_scaled), size=min(500, len(X_scaled)), replace=False)
plot_dendrogram(
    X_scaled[idx], filename='02_clustering_dendrogram_agglomerative',
)


In [ ]:
from clustering import run_agglomerative

agg_labels, agg_model = run_agglomerative(X_scaled, n_clusters=best_k)
agg_metrics = compute_cluster_metrics(X_scaled, agg_labels, 'Agglomerative')
print(agg_metrics)


## 🔴 Step 7 — DBSCAN with auto-chosen eps

DBSCAN is density-based — it doesn't need `k`. Instead it needs:
* `eps`: how close points must be to count as neighbours
* `min_samples`: how many neighbours form a dense region

The **k-distance plot** below sorts every point by its distance to its 5th nearest neighbour. The "elbow" of the curve suggests a good `eps`.


In [ ]:
from clustering import suggest_dbscan_eps, run_dbscan
from evaluation import plot_dbscan_kdistance

# Step 7a — k-distance plot to choose eps
distances, suggested_eps = suggest_dbscan_eps(X_scaled, k=5)
plot_dbscan_kdistance(
    distances, suggested_eps,
    filename='02_clustering_dbscan_eps_selection',
    k=5,
)
print(f"Auto-suggested eps = {suggested_eps:.3f}")


In [ ]:
# Step 7b — fit DBSCAN
dbscan_labels, dbscan_model = run_dbscan(X_scaled, eps=suggested_eps, min_samples=5)
dbscan_metrics = compute_cluster_metrics(X_scaled, dbscan_labels, 'DBSCAN')
print(dbscan_metrics)


## 📊 Step 8 — Compare the three algorithms

Internal validity metrics (no ground truth needed):

| Metric | Range | Better when |
|---|---|---|
| **Silhouette** | [-1, +1] | Higher |
| **Davies-Bouldin** | [0, ∞) | Lower |
| **Calinski-Harabasz** | [0, ∞) | Higher |


In [ ]:
from clustering import metrics_to_dataframe
from evaluation import save_results_table, plot_metric_comparison

metrics_df = metrics_to_dataframe([kmeans_metrics, agg_metrics, dbscan_metrics])

# Save the comparison table
save_results_table(metrics_df, '02_clustering_metrics_comparison')

# Plot the three internal metrics side by side
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ['silhouette', 'davies_bouldin', 'calinski_harabasz']):
    axes_data = metrics_df.set_index('algorithm')[metric]
    axes_data.plot.bar(ax=ax, color=['#3498db', '#e74c3c', '#2ecc71'])
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()

# Manually save with the naming convention
from evaluation import FIGURES_DIR
fig.savefig(FIGURES_DIR / '02_clustering_metrics_comparison.png')
print(f"  📊 Saved: results/figures/02_clustering_metrics_comparison.png")

metrics_df


## 🌌 Step 9 — Visualize clusters in 2D (PCA)

We can't visualize 17 dimensions directly, so project to 2D with PCA. Then color by cluster label for each of the three algorithms.


In [ ]:
from sklearn.decomposition import PCA
from evaluation import plot_clusters_2d

pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X_scaled)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

plot_clusters_2d(
    X_2d, kmeans_labels,
    filename='02_clustering_kmeans_pca_visualization',
    title=f'K-Means (k={best_k}) — PCA projection',
)


## 👤 Step 10 — Cluster profiles (the report's narrative)

What does each cluster *mean*? Compute the average feature value per cluster — this is what we'll describe in the report. The cluster with the highest `PURCHASES` is your "active shoppers", etc.


In [ ]:
from clustering import profile_clusters

# Use the un-scaled dataframe (drop the ID) so means are interpretable
df_for_profile = df.drop(columns=['CUST_ID']) if 'CUST_ID' in df.columns else df.copy()
df_for_profile = df_for_profile.fillna(df_for_profile.median(numeric_only=True))

profile = profile_clusters(df_for_profile, kmeans_labels)
save_results_table(profile, '02_clustering_kmeans_profiles')
profile.round(2)


## ✅ Step 11 — What we saved

Check the `results/` folder. You should see:

**Figures (`results/figures/`):**
- `02_clustering_elbow_method.png`
- `02_clustering_silhouette_kmeans.png`
- `02_clustering_dendrogram_agglomerative.png`
- `02_clustering_dbscan_eps_selection.png`
- `02_clustering_metrics_comparison.png`
- `02_clustering_kmeans_pca_visualization.png`

**Tables (`results/tables/`):**
- `02_clustering_metrics_comparison.csv`
- `02_clustering_kmeans_profiles.csv`

These exact names will appear in the report. Reference them directly — no renaming needed.


In [ ]:
!ls -la results/figures/ results/tables/


## 🧠 Insights for the report

Things to look for when interpreting the output (helpful for the demo and report):

1. **Which algorithm scored best?** Compare silhouette / DB / CH scores. K-Means usually wins on silhouette for well-separated data; DBSCAN wins when there are irregular shapes.
2. **How many noise points did DBSCAN flag?** A high count means data has many outliers — interesting business insight.
3. **What do the cluster profiles tell you?** Looking at the means: high-spenders vs. low-spenders, high-balance vs. low-balance, etc. The report should describe each cluster in plain English (≤ 2 sentences each).
4. **Did the dendrogram suggest a natural number of clusters?** The largest vertical gap in the dendrogram = the most natural cluster split. Often matches the elbow.

These map directly to **Hypothesis H4** in our Introduction (DBSCAN finding non-spherical structure missed by K-Means).
